# CLIP 对比语言-图像预训练模型代码精读

> **论文**：*Learning Transferable Visual Models From Natural Language Supervision*（OpenAI, 2021）  
> **核心思想**：以 **对比学习** 为目标，在 4 亿图文对上联合训练图像编码器（ViT）和文本编码器（Transformer），使同一语义的图文特征在嵌入空间中彼此靠近，不同语义的远离。  
> **零样本能力**：训练后无需微调，仅通过文本 Prompt 即可完成图像分类、检索等任务。

## 一、环境依赖检查 & CLIP 官方 API 快速推理

### 1.1 环境检查

确认 torch、torchvision、transformers 版本，以及 CUDA 可用性。

In [1]:
!pip list|grep torch

torch                                 2.6.0+cu124
torchao                               0.10.0
torchaudio                            2.6.0+cu124
torchdata                             0.11.0
torchsummary                          1.5.1
torchtune                             0.6.1
torchvision                           0.21.0+cu124


In [4]:
!pip list|grep transformers

sentence-transformers                 4.1.0
transformers                          4.53.2


### 1.2 CLIP 官方 API 快速推理演示

加载 `openai/clip-vit-base-patch32`，对一张图像和三段候选文本做零样本分类推理，
输出各文本与图像的匹配概率分布。

In [2]:
from transformers import CLIPProcessor, CLIPModel  # 从HuggingFace导入CLIP模型类和处理器类
from PIL import Image  # 导入Pillow库，用于加载本地图像文件
CACHE_DIR="./model_cache/CLIP"

# 从HuggingFace Hub加载预训练的CLIP模型（ViT-B/32版本：ViT backbone，patch_size=32）
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32",cache_dir=CACHE_DIR)  # 返回 CLIPModel 实例，含 text_model、vision_model、投影层、logit_scale 四个子模块
# 加载对应处理器：包含图像处理器（resize/归一化）和文本分词器（BPE tokenizer）
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32",cache_dir=CACHE_DIR)  # 返回 CLIPProcessor，内含 image_processor（归一化/resize）和 tokenizer（BPE分词）
print(model)  # 打印CLIP完整模型结构（含text_model、vision_model、投影层、logit_scale）
print('-'*100)  # 打印分隔线，分隔模型结构与处理器信息
print(processor)  # 打印处理器配置（图像归一化均值/方差和tokenizer词表大小等信息）

# 加载本地图像（luke.jpg），供推理使用
image = Image.open('cat.png')  # 返回 PIL.Image 对象（RGB格式），后续由处理器 resize 至 224×224 并归一化

# 定义候选文本描述列表，用于零样本分类（Zero-Shot Classification）
texts = ["a photo of a cat", "a photo of a dog", "a photo of a girl"]  # 3条候选描述，类型 List[str]，模型计算每条与图像的余弦相似度

# 处理器同时预处理文本和图像：
# - text部分：BPE分词 → input_ids（token整数序列）+ attention_mask（有效位为1/padding位为0），形状均为 [3, max_len]
# - image部分：resize到224×224、归一化（均值/方差来自预训练设置）→ pixel_values，形状为 [1, 3, 224, 224]
# padding=True：将3段文本填充到相同长度（max_len为最长那段分词后的长度）
inputs = processor(text=texts, images=image, return_tensors="pt", padding=True)  # 返回 BatchFeature（类dict），键为 input_ids/attention_mask/pixel_values
print(inputs)  # 打印预处理后的输入字典（各张量键名和形状）

# 执行CLIP前向传播：文本和图像分别经过各自编码器，再经投影层对齐到同一嵌入空间
outputs = model(**inputs)  # 返回 CLIPOutput，含 logits_per_image(1,3)、logits_per_text(3,1)、text_embeds(3,512)、image_embeds(1,512)
logits_per_image = outputs.logits_per_image  # 图像对每段文本的相似度得分，类型 Tensor，形状 (1, 3)：1张图像 × 3段文本，值越大越匹配
probs = logits_per_image.softmax(dim=1)  # 对3段文本的得分沿dim=1做softmax，得到概率分布，形状 (1, 3)，3个值之和严格为1

# 打印最终的概率分布：最高概率对应的文本为模型认为最匹配的描述
print("Probabilities:", probs)  # 打印形状 (1, 3) 的概率张量，每列对应一段候选文本的匹配概率

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((51

## 二、CLIP 官方接口深度解析

### 2.1 前向传播过程详细分解

逐步拆解 CLIP 前向传播的每个阶段，打印各中间张量的形状，理解从输入到输出的完整数据流：
- 文本编码 → 文本特征投影 → L2归一化
- 图像编码 → 图像特征投影 → L2归一化  
- 相似度矩阵计算 → softmax 概率分布 → 最优匹配

### 2.2 L2 归一化原理详解

**L2 归一化（L2 Normalization）** 是将一个向量除以其 **L2 范数（欧几里得范数）**，使得归一化后的向量长度（模）为 1，即将向量投影到单位超球面上。

**公式：**

$$\hat{x} = \frac{x}{\|x\|_2} = \frac{x}{\sqrt{\sum_{i} x_i^2}}$$

---

#### 在 CLIP 中的作用

1. **图像编码器** 输出图像特征向量 → L2 归一化 → 单位向量
2. **文本编码器** 输出文本特征向量 → L2 归一化 → 单位向量
3. 归一化后，两个向量的**点积**等价于**余弦相似度**：

$$\cos(\theta) = \hat{x}_{image} \cdot \hat{x}_{text}$$

---

#### 为什么要做 L2 归一化？

| 原因 | 说明 |
|------|------|
| 消除模长影响 | 只关注方向（语义），不受向量大小干扰 |
| 点积 = 余弦相似度 | 计算简单高效，便于对比学习 |
| 数值稳定 | 防止特征值过大导致梯度爆炸 |
| 对比损失需要 | InfoNCE Loss 依赖余弦相似度矩阵 |

---

#### PyTorch 实现

```python
import torch.nn.functional as F

# p=2 表示 L2 范数，dim=-1 表示沿最后一个维度（嵌入维度）归一化
# 输入形状：(batch_size, embed_dim)
# 输出形状：(batch_size, embed_dim)，每行向量的 L2 范数 = 1
features = F.normalize(features, p=2, dim=-1)
```

> **注意**：在 `CLIPModel` 的完整前向传播中，`outputs.text_embeds` 和 `outputs.image_embeds` 已经是 L2 归一化后的单位向量；而手动调用 `model.text_projection(...)` 时，需要自行做 L2 归一化。

In [4]:
from transformers import CLIPProcessor, CLIPModel  # 从HuggingFace导入CLIP模型类和处理器类
from PIL import Image  # 导入Pillow库，用于加载图像文件
import torch  # 导入PyTorch，提供张量操作和模型推理能力

# 从HuggingFace Hub加载预训练CLIP模型（ViT-B/32：图像编码器使用32×32 patch的ViT）
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32",cache_dir=CACHE_DIR)  # 返回 CLIPModel 实例，含 text_model(12层Transformer)、vision_model(ViT-B/32)、投影层和 logit_scale
# 加载对应的CLIP处理器（负责图像预处理和文本BPE分词）
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32",cache_dir=CACHE_DIR)  # 返回 CLIPProcessor，内含 image_processor（resize/归一化）和 tokenizer（BPE，词表大小49408）

print("CLIP Model Architecture:")  # 提示即将打印模型结构
print(model)  # 打印完整CLIP模型结构（含text_model、vision_model、投影层、logit_scale）
print('-'*100)  # 打印分隔线，分隔模型结构与其他信息
print('='*100)  # 打印双重分隔线，标记新的信息块

image = Image.open('cat.png')  # 加载本地图像文件，返回 PIL.Image 对象（RGB格式），供处理器预处理使用

# 定义三段候选文本，用于图文相似度零样本分类
texts = ["a photo of a cat", "a photo of a dog", "a photo of a girl"]  # 3条候选描述，类型 List[str]，每条将被BPE分词后与图像计算余弦相似度

# 使用处理器同时预处理文本（BPE分词）和图像（resize/归一化），padding=True使文本等长
inputs = processor(text=texts, images=image, return_tensors="pt", padding=True)  # 返回 BatchFeature（类dict），含 input_ids(3,max_len)、attention_mask(3,max_len)、pixel_values(1,3,224,224)
print("Processed Inputs:")  # 提示打印预处理后的输入信息
print(f"Input keys: {list(inputs.keys())}")  # 打印输入字典的所有键名，如 ['input_ids', 'attention_mask', 'pixel_values']
for key, value in inputs.items():  # 遍历每个输入张量，打印其形状或类型
    if isinstance(value, torch.Tensor):  # 如果是张量，打印形状
        print(f"  {key}: {value.shape}")  # 格式：键名: 张量形状，如 input_ids: torch.Size([3, 7])
    else:  # 如果不是张量（如列表），打印类型
        print(f"  {key}: {type(value)}")  # 打印非张量类型，如 <class 'list'>
print('-'*50)  # 打印分隔线，分隔输入信息与前向传播分析

model.eval()  # 将模型设为评估模式：关闭Dropout，固定BatchNorm统计量，影响推理结果确定性

with torch.no_grad():  # 推理阶段关闭梯度计算，节省显存和计算资源（无需反向传播）
    print("Forward Pass Analysis:")  # 提示开始逐步分析前向传播
    print('-'*50)  # 打印分隔线

    # ============ 阶段1：文本编码 ============
    print("1. Text Encoding:")  # 打印阶段标题
    text_inputs = {  # 构造仅含文本相关键的输入字典，供 text_model 单独调用
        'input_ids': inputs['input_ids'],          # 文本token ID，形状 (3, max_len)：3段文本，每段填充至最大长度
        'attention_mask': inputs['attention_mask']  # 注意力掩码，形状 (3, max_len)：有效token位置为1，padding位置为0
    }  # text_inputs 类型 dict，键为字符串，值为 Tensor
    print(f"   Text input_ids shape: {text_inputs['input_ids'].shape}")  # 打印文本token ID张量形状：(num_texts, max_seq_len)
    print(f"   Text attention_mask shape: {text_inputs['attention_mask'].shape}")  # 打印注意力掩码形状：(num_texts, max_seq_len)

    # 通过CLIP的text_model（12层Transformer）编码文本
    text_outputs = model.text_model(**text_inputs)  # 返回 BaseModelOutputWithPooling：last_hidden_state(3,max_len,512) + pooler_output(3,512)
    # 通过文本投影头将隐藏维度（512）映射到共享嵌入空间（512，与图像特征维度对齐）
    text_embeds = model.text_projection(text_outputs.pooler_output)  # 对 [EOS] 位置特征做线性投影，返回 Tensor，形状 (3, 512)

    # 打印文本编码器各阶段输出形状
    print(f"   Text model hidden states shape: {text_outputs.last_hidden_state.shape}")  # 形状 (3, max_len, 512)：3段文本，每位置512维隐藏状态
    print(f"   Text model pooler output shape: {text_outputs.pooler_output.shape}")  # 形状 (3, 512)：取每段文本[EOS]位置的特征作为句子表示
    print(f"   Text embeddings (after projection) shape: {text_embeds.shape}")  # 形状 (3, 512)：经线性投影映射到共享嵌入空间
    print()  # 空行分隔

    # ============ 阶段2：图像编码 ============
    print("2. Vision Encoding:")  # 打印阶段标题
    vision_inputs = {  # 构造仅含图像相关键的输入字典，供 vision_model 单独调用
        'pixel_values': inputs['pixel_values']  # 预处理后的图像张量，形状 (1, 3, 224, 224)：1张图，RGB三通道，224×224分辨率
    }  # vision_inputs 类型 dict
    print(f"   Vision pixel_values shape: {vision_inputs['pixel_values'].shape}")  # 打印图像张量形状：(batch_size, channels, height, width)

    # 通过CLIP的vision_model（ViT-B/32，12层Transformer，patch_size=32）编码图像
    vision_outputs = model.vision_model(**vision_inputs)  # 返回 BaseModelOutputWithPooling：last_hidden_state(1,50,768) + pooler_output(1,768)
    # 通过视觉投影头将ViT隐藏维度（768）映射到共享嵌入空间（512）
    image_embeds = model.visual_projection(vision_outputs.pooler_output)  # 对 [CLS] 位置特征做线性投影（768→512），返回 Tensor，形状 (1, 512)

    # 打印视觉编码器各阶段输出形状
    print(f"   Vision model hidden states shape: {vision_outputs.last_hidden_state.shape}")  # 形状 (1, 50, 768)：50=1([CLS])+49(7×7 patches)，每位置768维
    print(f"   Vision model pooler output shape: {vision_outputs.pooler_output.shape}")  # 形状 (1, 768)：取 [CLS] token 的特征作为图像全局表示
    print(f"   Image embeddings (after projection) shape: {image_embeds.shape}")  # 形状 (1, 512)：经线性投影从768维映射到512维共享嵌入空间
    print()  # 空行分隔

    # ============ 阶段3：完整前向传播（一次性执行所有步骤）============
    print("3. Complete Forward Pass:")  # 打印阶段标题
    outputs = model(**inputs)  # 一次性前向传播，自动完成编码、投影、L2归一化和相似度计算，返回 CLIPOutput

    print(f"   Final text features shape: {outputs.text_embeds.shape}")  # 最终文本特征形状 (3, 512)，已经过L2归一化（单位向量）
    print(f"   Final image features shape: {outputs.image_embeds.shape}")  # 最终图像特征形状 (1, 512)，已经过L2归一化（单位向量）
    print(f"   Logits per image shape: {outputs.logits_per_image.shape}")  # 图像对文本的得分矩阵，形状 (1, 3)：1张图像 × 3段文本
    print(f"   Logits per text shape: {outputs.logits_per_text.shape}")  # 文本对图像的得分矩阵，形状 (3, 1)：3段文本 × 1张图像
    print()  # 空行分隔

    # ============ 阶段4：相似度计算原理解析 ============
    print("4. Similarity Computation Details:")  # 打印阶段标题
    logits_per_image = outputs.logits_per_image  # 图像对各文本的相似度得分，类型 Tensor，形状 (1,3)，值=logit_scale × 余弦相似度
    logits_per_text = outputs.logits_per_text    # 文本对图像的相似度得分，类型 Tensor，形状 (3,1)，为 logits_per_image 的转置

    print(f"   Image-to-text logits: {logits_per_image.shape}")  # 打印图像→文本方向的logits形状：(num_images, num_texts)
    print(f"   Text-to-image logits: {logits_per_text.shape}")  # 打印文本→图像方向的logits形状：(num_texts, num_images)
    # logit_scale 是可学习温度参数，exp后约为100，放大余弦相似度使分布更尖锐，便于分类
    print(f"   Logit scale parameter: {model.logit_scale.exp().item():.4f}")  # 打印温度参数exp值（标量），越大分布越尖锐

    # 对图像→文本方向的logits做softmax（dim=1），得到每张图像匹配各文本的概率
    probs_image_to_text = logits_per_image.softmax(dim=1)  # 沿文本维度softmax，返回 Tensor，形状 (1, 3)，每行3个概率值之和为1
    # 对文本→图像方向的logits做softmax（dim=0），得到每段文本匹配各图像的概率
    probs_text_to_image = logits_per_text.softmax(dim=0)  # 沿图像维度softmax，返回 Tensor，形状 (3, 1)，每列各概率值之和为1

    print(f"   Image-to-text probabilities shape: {probs_image_to_text.shape}")  # 打印图→文概率张量形状：(num_images, num_texts)
    print(f"   Text-to-image probabilities shape: {probs_text_to_image.shape}")  # 打印文→图概率张量形状：(num_texts, num_images)
    print()  # 空行分隔

    # ============ 阶段5：详细结果展示 ============
    print("5. Detailed Results:")  # 打印阶段标题
    print('-'*30)  # 打印分隔线

    print("Raw similarity scores (logits):")  # 提示打印原始相似度得分（softmax之前的原始logit值）
    for i, text in enumerate(texts):  # 遍历每段文本（i为索引，text为文本内容），打印对应的原始logit值
        print(f"   '{text}': {logits_per_image[0, i].item():.4f}")  # 取第0张图像对第i段文本的得分，.item()将标量张量转为Python浮点数
    print()  # 空行分隔

    print("Probability distribution:")  # 提示打印softmax概率分布
    probs = probs_image_to_text[0]  # 取第一张（也是唯一一张）图像的概率向量，类型 Tensor，形状 (3,)，3个元素对应3段文本
    for i, text in enumerate(texts):  # 遍历每段文本，打印其匹配概率和百分比形式
        print(f"   '{text}': {probs[i].item():.4f} ({probs[i].item()*100:.2f}%)")  # 打印概率值（4位小数）和百分比（2位小数）
    print()  # 空行分隔

    best_match_idx = torch.argmax(probs).item()  # 找到概率最高的文本的索引（int），对应最匹配的候选文本
    # 打印最终预测结果：最匹配的文本描述及其对应概率
    print(f"Best match: '{texts[best_match_idx]}' with probability {probs[best_match_idx].item():.4f}")  # 打印最佳匹配文本及其概率
    print()  # 空行分隔

# ============ 阶段6：模型参数统计 ============
print("6. Model Parameter Statistics:")  # 打印阶段标题
print('-'*40)  # 打印分隔线
total_params = sum(p.numel() for p in model.parameters())  # 统计模型全部参数量（int），p.numel()返回每个参数张量的元素总数
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)  # 统计可训练参数量（int），requires_grad=True表示该参数参与梯度更新

print(f"Total parameters: {total_params:,}")  # 打印总参数量，使用逗号格式化（如 151,277,313）便于阅读
print(f"Trainable parameters: {trainable_params:,}")  # 打印可训练参数量，与总量相同时说明所有参数均参与训练

# 分别统计各子模块的参数量，用于了解模型各部分的参数分布
text_params = sum(p.numel() for p in model.text_model.parameters())  # 文本Transformer（12层）总参数量（int）
vision_params = sum(p.numel() for p in model.vision_model.parameters())  # 视觉Transformer/ViT（12层）总参数量（int）
text_proj_params = sum(p.numel() for p in model.text_projection.parameters())  # 文本投影头（Linear层）参数量（int），实现512→512映射
visual_proj_params = sum(p.numel() for p in model.visual_projection.parameters())  # 视觉投影头（Linear层）参数量（int），实现768→512映射

print(f"Text model parameters: {text_params:,}")  # 打印文本模型参数量
print(f"Vision model parameters: {vision_params:,}")  # 打印视觉模型参数量
print(f"Text projection parameters: {text_proj_params:,}")  # 打印文本投影层参数量（512×512=262,144）
print(f"Visual projection parameters: {visual_proj_params:,}")  # 打印视觉投影层参数量（768×512=393,216）
print(f"Logit scale parameter: 1")  # logit_scale 是单个可学习标量（Tensor，shape=()），参数量为1

print('='*100)  # 打印结束分隔线
print("Analysis Complete!")  # 打印完成提示

CLIP Model Architecture:
CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (la

## 三、手动实现 CLIP 模型

本节从零手动实现简化版 CLIP 模型，加深对对比学习原理的理解。
- **3.1 节**：最简线性编码器 + InfoNCE 损失，验证核心数学逻辑
- **3.2 节**：引入完整 ViT 图像编码器（PatchEmbedding + MultiHeadAttention），贴近真实实现

> **关键知识点**：相似度矩阵在 batch 内计算，论文中的 N 就是 batch size，
> 对角线元素为正样本对（图像 i 与文本 i），其余为负样本对。

> **注意**：相似度矩阵是在 batch 内计算的，论文中的 N 就是 batch size。对角线元素为正样本对（图像 i 与文本 i），其余元素均为负样本对。

### 3.1 简单 CLIP（线性编码器版本）

使用最简单的线性层作为图像编码器（输入维度1024）和文本编码器（输入维度512），
演示 CLIP 对比学习的核心损失计算流程，验证相似度矩阵形状和 InfoNCE 损失值是否正确。

In [ ]:
import torch  # 导入PyTorch库
import torch.nn as nn  # 导入PyTorch神经网络模块
import torch.nn.functional as F  # 导入PyTorch函数式API

# 定义简单的图像编码器和文本编码器
class ImageEncoder(nn.Module):  # 定义图像编码器类，继承自nn.Module
    """
    简单线性图像编码器。

    使用单层线性层将输入图像特征（固定1024维）线性映射到共享嵌入空间（embed_dim维），
    是最简化的图像编码器实现，主要用于验证CLIP对比学习的核心数学逻辑和损失函数正确性。
    """

    def __init__(self, embed_dim):  # 初始化方法，接收嵌入维度参数
        """
        初始化图像编码器。

        参数：
            embed_dim (int): 输出嵌入向量的维度大小，与文本编码器输出维度保持一致，
                             确保图文特征可在同一共享嵌入空间内进行余弦相似度计算。
        """
        super(ImageEncoder, self).__init__()  # 调用父类初始化方法
        self.fc = nn.Linear(1024, embed_dim)  # 假设输入图像特征维度为 1024，创建线性层将其映射到嵌入维度

    def forward(self, x):  # 前向传播方法
        """
        前向传播：将图像特征线性映射到嵌入空间。

        参数：
            x (torch.Tensor): 输入图像特征，形状 (batch_size, 1024)。
                               - batch_size: 批次中的样本数量
                               - 1024: 预设的输入图像特征维度

        返回：
            torch.Tensor: 图像嵌入向量，形状 (batch_size, embed_dim)。
                          - batch_size: 批次中的样本数量
                          - embed_dim: 共享嵌入空间维度（与文本编码器输出维度相同）
        """
        return self.fc(x)  # 返回线性变换后的结果

class TextEncoder(nn.Module):  # 定义文本编码器类，继承自nn.Module
    """
    简单线性文本编码器。

    使用单层线性层将输入文本特征（固定512维）线性映射到共享嵌入空间（embed_dim维），
    与图像编码器配合使用，共同构成最简化的CLIP对比学习框架。
    """

    def __init__(self, embed_dim):  # 初始化方法，接收嵌入维度参数
        """
        初始化文本编码器。

        参数：
            embed_dim (int): 输出嵌入向量的维度大小，与图像编码器输出维度保持一致，
                             确保图文特征可在同一共享嵌入空间内进行余弦相似度计算。
        """
        super(TextEncoder, self).__init__()  # 调用父类初始化方法
        self.fc = nn.Linear(512, embed_dim)  # 假设输入文本特征维度为 512，创建线性层将其映射到嵌入维度

    def forward(self, x):  # 前向传播方法
        """
        前向传播：将文本特征线性映射到嵌入空间。

        参数：
            x (torch.Tensor): 输入文本特征，形状 (batch_size, 512)。
                               - batch_size: 批次中的样本数量
                               - 512: 预设的输入文本特征维度

        返回：
            torch.Tensor: 文本嵌入向量，形状 (batch_size, embed_dim)。
                          - batch_size: 批次中的样本数量
                          - embed_dim: 共享嵌入空间维度（与图像编码器输出维度相同）
        """
        return self.fc(x)  # 返回线性变换后的结果

# 定义 CLIP 模型
class CLIP(nn.Module):  # 定义CLIP模型类，继承自nn.Module
    """
    简化版CLIP对比学习模型（线性编码器版本）。

    将图像编码器和文本编码器组合成完整的CLIP框架：
    1. 分别编码图像和文本到同一共享嵌入空间
    2. L2归一化后计算余弦相似度矩阵（形状 N×N，N为batch size）
    3. 乘以可学习温度参数 logit_scale 得到最终 logits

    本版本使用最简单的线性编码器，主要用于验证对比学习的核心数学逻辑。
    """

    def __init__(self, embed_dim):  # 初始化方法，接收嵌入维度参数
        """
        初始化CLIP模型。

        参数：
            embed_dim (int): 共享嵌入空间的维度大小，图像和文本特征在此维度下对齐，
                             用于余弦相似度计算。
        """
        super(CLIP, self).__init__()  # 调用父类初始化方法
        self.image_encoder = ImageEncoder(embed_dim)  # 创建图像编码器实例
        self.text_encoder = TextEncoder(embed_dim)  # 创建文本编码器实例
        self.logit_scale = nn.Parameter(torch.ones([]) * 2.6592)  # 可学习的温度参数，初始化为2.6592

    def forward(self, images, texts):  # 前向传播方法，接收图像和文本输入
        """
        前向传播：计算图像与文本的跨模态相似度矩阵。

        参数：
            images (torch.Tensor): 输入图像特征，形状 (batch_size, 1024)。
                                   - batch_size: 批次中的样本数量（即论文中的N）
                                   - 1024: 图像编码器输入维度
            texts (torch.Tensor): 输入文本特征，形状 (batch_size, 512)。
                                  - batch_size: 批次中的样本数量（与图像数量相同）
                                  - 512: 文本编码器输入维度

        返回：
            tuple(torch.Tensor, torch.Tensor):
                - logits_per_image: 图像对所有文本的相似度得分，形状 (batch_size, batch_size)，
                                    第i行第j列表示第i张图像与第j段文本的相似度，对角线为正样本对。
                - logits_per_text: 文本对所有图像的相似度得分，形状 (batch_size, batch_size)，
                                   为 logits_per_image 的转置矩阵。
        """
        # 编码图像和文本
        image_features = self.image_encoder(images)  # 调用线性图像编码器，返回 Tensor，形状 (batch_size, embed_dim)，即图像特征嵌入向量
        text_features = self.text_encoder(texts)  # 调用线性文本编码器，返回 Tensor，形状 (batch_size, embed_dim)，即文本特征嵌入向量

        # 归一化特征：使每个特征向量的L2模长=1，余弦相似度等价于点积
        image_features = F.normalize(image_features, dim=-1)  # 沿嵌入维度做L2归一化，形状不变仍为 (batch_size, embed_dim)，每行向量的L2范数=1
        text_features = F.normalize(text_features, dim=-1)  # 沿嵌入维度做L2归一化，形状不变仍为 (batch_size, embed_dim)，每行向量的L2范数=1

        # 计算相似度矩阵：logit_scale放大余弦相似度，使分布更尖锐，便于分类
        logit_scale = self.logit_scale.exp()  # 对可学习温度参数取指数，返回标量 Tensor；初始值 exp(2.6592)≈14.3，训练后约为100
        logits_per_image = logit_scale * image_features @ text_features.t()  # 矩阵乘法计算图文余弦相似度并缩放，返回 Tensor，形状 (batch_size, batch_size)：第[i,j]元素=第i张图像与第j段文本的相似度得分
        logits_per_text = logits_per_image.t()  # 转置得到文本→图像方向的相似度矩阵，返回 Tensor，形状 (batch_size, batch_size)：第[j,i]元素=第j段文本与第i张图像的相似度得分

        return logits_per_image, logits_per_text  # 返回 tuple(Tensor,Tensor)：图像→文本得分矩阵(batch,batch) 和 文本→图像得分矩阵(batch,batch)

# 定义对比损失函数
def clip_loss(logits_per_image, logits_per_text):  # 定义CLIP损失函数，接收两个相似度矩阵
    """
    计算CLIP对比损失（InfoNCE Loss / 对称交叉熵损失）。

    对图像→文本和文本→图像两个方向分别计算交叉熵损失，然后取平均作为总损失。
    标签为对角线索引 (0, 1, 2, ..., N-1)，因为第i张图像与第i段文本互为正样本对。

    参数：
        logits_per_image (torch.Tensor): 图像对所有文本的相似度得分，形状 (batch_size, batch_size)。
                                         - 第i行第j列：第i张图像与第j段文本的相似度分数
                                         - 对角线元素为正样本对，其余为负样本对
        logits_per_text (torch.Tensor): 文本对所有图像的相似度得分，形状 (batch_size, batch_size)，
                                        即 logits_per_image 的转置矩阵。

    返回：
        torch.Tensor: 标量张量，CLIP对比损失值（图像→文本损失与文本→图像损失的平均值）。
    """
    # 计算图像到文本的损失
    labels = torch.arange(logits_per_image.size(0)).to(logits_per_image.device)  # 生成标签 [0,1,...,N-1]，类型 Tensor(long)，形状 (batch_size,)；第i个值=i表示第i张图像的正样本是第i段文本
    loss_image = F.cross_entropy(logits_per_image, labels)  # 以每行（图像）的所有文本得分为logit、labels为目标类别计算交叉熵，返回标量 Tensor（图像→文本方向的平均损失）

    # 计算文本到图像的损失
    loss_text = F.cross_entropy(logits_per_text, labels)  # 以每行（文本）的所有图像得分为logit、labels为目标类别计算交叉熵，返回标量 Tensor（文本→图像方向的平均损失）

    # 总损失：对两个方向取均值，实现对称对比损失
    total_loss = (loss_image + loss_text) / 2  # 两方向损失求平均，返回标量 Tensor，这正是论文公式中的对称 InfoNCE 损失
    return total_loss  # 返回标量 Tensor，即最终的CLIP对比损失值，用于反向传播更新参数

# 示例数据
batch_size = 4  # 批次大小（int）：一次前向传播包含4组图文对，即论文中的 N=4
embed_dim = 64  # 共享嵌入空间维度（int）：图像和文本特征均被映射到64维空间
images = torch.randn(batch_size, 1024)  # 随机生成图像特征张量，类型 Tensor，形状 (4, 1024)：4张图像，每张1024维特征（替代真实CNN/ViT输出）
texts = torch.randn(batch_size, 512)    # 随机生成文本特征张量，类型 Tensor，形状 (4, 512)：4段文本，每段512维特征（替代真实Tokenizer+Transformer输出）

# 初始化模型
model = CLIP(embed_dim)  # 创建CLIP模型实例，返回 CLIP 对象，嵌入维度为64，内含线性图像编码器和线性文本编码器

# 前向传播
logits_per_image, logits_per_text = model(images, texts)  # 执行模型前向传播，返回两个 Tensor：logits_per_image 形状(4,4)、logits_per_text 形状(4,4)，均为图文相似度得分矩阵
print("logits_per_image:", logits_per_image.shape)  # 打印图像→文本相似度矩阵的形状，期望输出 torch.Size([4, 4])：4张图像×4段文本
print("logits_per_text:", logits_per_text.shape)  # 打印文本→图像相似度矩阵的形状，期望输出 torch.Size([4, 4])：4段文本×4张图像
# 计算损失
loss = clip_loss(logits_per_image, logits_per_text)  # 计算对称InfoNCE损失，返回标量 Tensor，值约为 log(4)≈1.386（随机初始化时各候选均等概率）
print("Loss:", loss.item())  # 打印标量损失值（Python float），.item()将标量Tensor转为Python原生浮点数

logits_per_image: torch.Size([4, 4])
logits_per_text: torch.Size([4, 4])
Loss: 1.390106439590454


### 3.2 带完整 ViT 图像编码器的 CLIP 实现

使用 `PatchEmbedding + MultiHeadAttention + ImageEncoder` 手动搭建完整 ViT 视觉编码器，
同时将文本编码器升级为带自注意力机制的简单 Transformer，进一步贴近真实 CLIP 模型结构。

In [ ]:
import math  # 导入数学库，用于数学计算如平方根等
import torch  # 导入PyTorch库，用于深度学习模型构建和训练
import torch.nn as nn  # 导入PyTorch神经网络模块，提供构建神经网络的各种组件
import torch.nn.functional as F  # 导入PyTorch函数式API，提供各种激活函数和损失函数等

class PatchEmbedding(nn.Module):  # 定义图像分块嵌入类，将图像分成小块并嵌入到向量空间
    """
    图像分块嵌入层（Patch Embedding）。

    将输入图像切分为若干固定大小的 patch，通过卷积层将每个 patch 线性映射为嵌入向量，
    再拼接可学习的 [CLS] token，并加上可学习的位置编码，作为 ViT Transformer 编码器的输入。
    对应论文中"将图像转化为 token 序列"的核心步骤。
    """

    def __init__(self, image_size=224, patch_size=16, in_channels=3, embed_dim=768):  # 初始化方法，设置默认参数
        """
        初始化图像分块嵌入层。

        参数：
            image_size (int): 输入图像的高宽（默认224，即224×224像素）。
            patch_size (int): 每个 patch 的高宽（默认16，即16×16像素的小块）。
            in_channels (int): 输入图像的通道数（默认3，对应RGB三通道）。
            embed_dim (int): 每个 patch 映射后的嵌入维度（默认768，对应ViT-B/16）。
        """
        super().__init__()  # 调用父类初始化方法
        self.image_size = image_size  # 存储图像大小，默认为224x224像素
        self.patch_size = patch_size  # 存储patch的大小，用于将图像分割成固定大小的patch，patch_size通常为16，表示将图像分割成16x16大小的小块
        self.num_patches = (image_size // patch_size) ** 2  # 计算patch的总数量，等于(图像大小/patch大小)的平方

        # 将图像分成patch并进行线性投影，使用卷积层实现，卷积核大小和步长都等于patch_size
        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

        # 位置编码相关参数
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))  # 创建可学习的分类token，用于整体图像表示
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, embed_dim))  # 创建可学习的位置编码，+1是为了包含cls_token的位置,看下面forward即可理解

    def forward(self, x):  # 前向传播方法
        """
        前向传播：将图像转换为 patch token 序列并加入位置编码。

        参数：
            x (torch.Tensor): 输入图像张量，形状 (batch_size, in_channels, image_size, image_size)。
                               - batch_size: 批次中的图像数量
                               - in_channels: 图像通道数（3对应RGB）
                               - image_size: 图像高宽（224）

        返回：
            torch.Tensor: 带位置编码的 token 序列，形状 (batch_size, num_patches+1, embed_dim)。
                          - batch_size: 批次大小
                          - num_patches+1: patch数量+1个[CLS]token，num_patches=(image_size//patch_size)^2
                          - embed_dim: 每个 token 的嵌入维度
        """
        B = x.shape[0]  # 获取批次大小，类型 int，等于输入张量第0维的大小
        # 将图像转换为patch序列：卷积→(B,embed_dim,H/P,W/P)，flatten(2)→(B,embed_dim,N)，transpose→(B,N,embed_dim)
        x = self.projection(x).flatten(2).transpose(1, 2)  # 返回 Tensor，形状 (B, num_patches, embed_dim)：B为批次，num_patches=(H/P)*(W/P)，embed_dim为每patch嵌入维度

        # 添加分类token：将cls_token扩展到批次维度，然后与patch序列拼接
        cls_tokens = self.cls_token.expand(B, -1, -1)  # 扩展cls_token到批次大小
        x = torch.cat((cls_tokens, x), dim=1)  # 在序列维度上拼接cls_token和patch序列

        # 添加位置编码：将位置编码加到token序列上
        x = x + self.pos_embed  # 添加位置编码，提供序列中位置信息
        return x  # 返回处理后的序列

class MultiHeadAttention(nn.Module):  # 定义多头注意力机制类
    """
    多头自注意力机制（Multi-Head Self-Attention）。

    将输入序列通过 QKV 投影分别得到查询、键、值向量，
    按注意力头数分组后并行计算缩放点积注意力（Scaled Dot-Product Attention），
    最后拼接各头结果并通过输出投影层恢复原始维度。
    是 Transformer 编码器的核心组件。
    """

    def __init__(self, embed_dim, num_heads):  # 初始化方法，接收嵌入维度和注意力头数
        """
        初始化多头注意力层。

        参数：
            embed_dim (int): 输入序列的特征维度（即每个 token 的嵌入维度）。
            num_heads (int): 注意力头的数量，embed_dim 必须能被 num_heads 整除，
                             每个头的维度为 head_dim = embed_dim // num_heads。
        """
        super().__init__()  # 调用父类初始化方法
        self.num_heads = num_heads  # 存储注意力头数量
        self.head_dim = embed_dim // num_heads  # 计算每个注意力头的维度
        self.scale = self.head_dim ** -0.5  # 计算缩放因子，用于缩放注意力分数

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)  # 创建线性层，用于同时生成查询(Q)、键(K)和值(V)
        self.proj = nn.Linear(embed_dim, embed_dim)  # 创建输出投影层，将多头注意力的结果映射回原始维度

    def forward(self, x):  # 前向传播方法
        """
        前向传播：对输入 token 序列执行多头自注意力计算。

        参数：
            x (torch.Tensor): 输入 token 序列，形状 (batch_size, seq_len, embed_dim)。
                               - batch_size: 批次大小
                               - seq_len: 序列长度（num_patches+1，含[CLS]token）
                               - embed_dim: 每个 token 的特征维度

        返回：
            torch.Tensor: 经过多头注意力加权后的 token 序列，形状与输入相同
                          (batch_size, seq_len, embed_dim)。
        """
        B, N, C = x.shape  # 获取输入维度：B=批次大小(int)，N=序列长度即num_patches+1(int)，C=特征维度即embed_dim(int)
        # 生成QKV并重塑维度：Linear→(B,N,3C)，reshape→(B,N,3,heads,head_dim)，permute→(3,B,heads,N,head_dim)
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)  # 返回 Tensor，形状 (3, B, num_heads, N, head_dim)：第0维的3对应Q/K/V三个矩阵
        q, k, v = qkv[0], qkv[1], qkv[2]  # 分离查询Q、键K、值V，每个形状均为 (B, num_heads, N, head_dim)

        # 计算注意力分数：查询和键的矩阵乘法，然后应用缩放
        attn = (q @ k.transpose(-2, -1)) * self.scale  # 计算缩放点积注意力
        attn = attn.softmax(dim=-1)  # 对注意力分数应用softmax归一化

        # 应用注意力权重到值上，并重塑回原始维度
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)  # 计算加权和并重塑维度
        x = self.proj(x)  # 通过输出投影层
        return x  # 返回注意力的输出

class ImageEncoder(nn.Module):  # 定义图像编码器类
    """
    基于 ViT 的图像编码器（Vision Transformer Image Encoder）。

    完整实现了 ViT 视觉编码器：
    1. PatchEmbedding：将图像切分为 patch，线性嵌入并加位置编码
    2. 多层 Transformer 块（LayerNorm + MultiHeadAttention + LayerNorm + MLP，均带残差连接）
    3. 最终 LayerNorm 后取 [CLS] token 输出作为整张图像的全局表示

    对应 CLIP 论文中的视觉编码器（ViT-B/16 默认配置）。
    """

    def __init__(self, embed_dim, image_size=224, patch_size=16, num_layers=12, num_heads=12, mlp_ratio=4.0):  # 初始化方法
        """
        初始化 ViT 图像编码器。

        参数：
            embed_dim (int): 每个 token 的嵌入维度，也是最终输出的图像特征维度。
            image_size (int): 输入图像的高宽（默认224，即224×224）。
            patch_size (int): 每个 patch 的高宽（默认16，即16×16）。
            num_layers (int): Transformer 编码器层的数量（默认12层）。
            num_heads (int): 多头注意力的头数（默认12头，每头维度=embed_dim//num_heads）。
            mlp_ratio (float): MLP 隐藏层维度相对于 embed_dim 的扩张倍数（默认4.0）。
        """
        super().__init__()  # 调用父类初始化方法
        # 创建图像分块嵌入层
        self.patch_embed = PatchEmbedding(image_size, patch_size, 3, embed_dim)  # 创建图像分块嵌入实例

        # Transformer编码器层：创建多个Transformer层
        self.layers = nn.ModuleList([])  # 创建模块列表，用于存储Transformer层
        for _ in range(num_layers):  # 循环创建指定数量的Transformer层
            layer = nn.Sequential(  # 使用Sequential容器组织层的顺序
                nn.LayerNorm(embed_dim),  # 第一个层归一化
                MultiHeadAttention(embed_dim, num_heads),  # 多头注意力层
                nn.LayerNorm(embed_dim),  # 第二个层归一化
                nn.Sequential(  # MLP块，包含两个线性层和GELU激活函数
                    nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),  # 第一个线性层，扩展维度
                    nn.GELU(),  # GELU激活函数
                    nn.Linear(int(embed_dim * mlp_ratio), embed_dim)  # 第二个线性层，恢复维度
                )
            )
            self.layers.append(layer)  # 将创建的层添加到模块列表中

        self.norm = nn.LayerNorm(embed_dim)  # 最终的层归一化

    def forward(self, x):  # 前向传播方法
        """
        前向传播：将图像编码为全局特征向量（[CLS] token）。

        参数：
            x (torch.Tensor): 输入图像张量，形状 (batch_size, 3, image_size, image_size)。
                               - batch_size: 批次大小
                               - 3: RGB 三通道
                               - image_size: 图像高宽（224）

        返回：
            torch.Tensor: 图像全局特征向量，形状 (batch_size, embed_dim)。
                          - batch_size: 批次大小
                          - embed_dim: 嵌入维度（取 [CLS] token 对应位置的特征作为整图表示）
        """
        x = self.patch_embed(x)  # 通过图像分块嵌入层处理输入

        for layer in self.layers:  # 遍历所有Transformer层
            x = x + layer[0](x)  # 第一个残差连接：原始输入 + 层归一化和注意力的输出
            x = x + layer[2](layer[3](layer[1](x)))  # 第二个残差连接：中间结果 + 层归一化和MLP的输出

        x = self.norm(x)  # 应用最终的层归一化
        return x[:, 0]  # 只返回[CLS]token的特征，用作整个图像的表示

class TextEncoder(nn.Module):  # 定义文本编码器类，继承自nn.Module
    """
    带简单自注意力机制的文本编码器。

    在线性层基础上引入了简单的自注意力机制：对输入文本特征计算 QKV 并执行缩放点积注意力，
    输出经过残差连接后通过最终线性层映射到共享嵌入空间。
    该实现为教学用简化版本，贴近但不完全等同于真实 CLIP 文本 Transformer。
    """

    def __init__(self, embed_dim):  # 初始化方法，接收嵌入维度参数
        """
        初始化文本编码器。

        参数：
            embed_dim (int): 最终输出嵌入向量的维度大小，与图像编码器输出维度保持一致，
                             确保图文特征可在同一共享嵌入空间内进行余弦相似度计算。
        """
        super(TextEncoder, self).__init__()  # 调用父类初始化方法
        self.fc = nn.Linear(512, embed_dim)  # 假设输入文本特征维度为 512，创建线性层将其映射到嵌入维度

    def forward(self, x):  # 前向传播方法
        """
        前向传播：通过自注意力机制对文本特征进行上下文建模，再映射到嵌入空间。

        参数：
            x (torch.Tensor): 输入文本特征，形状 (batch_size, 512)。
                               - batch_size: 批次中的样本数量
                               - 512: 预设的输入文本特征维度

        返回：
            torch.Tensor: 文本嵌入向量，形状 (batch_size, embed_dim)。
                          - batch_size: 批次中的样本数量
                          - embed_dim: 共享嵌入空间维度（与图像编码器输出维度相同）
        """
        # 实现类似CLIP模型中的注意力机制
        batch_size, seq_len = x.shape[0], x.shape[1]  # 获取批次大小和序列长度

        # 假设我们有一个简单的自注意力机制
        # 创建QKV投影
        hidden_size = 512  # 设置隐藏层大小
        q_proj = nn.Linear(seq_len, hidden_size).to(x.device)  # 创建查询投影层，并移动到与输入相同的设备上
        k_proj = nn.Linear(seq_len, hidden_size).to(x.device)  # 创建键投影层
        v_proj = nn.Linear(seq_len, hidden_size).to(x.device)  # 创建值投影层
        out_proj = nn.Linear(hidden_size, hidden_size).to(x.device)  # 创建输出投影层

        # 计算QKV
        q = q_proj(x)  # 生成查询向量
        k = k_proj(x)  # 生成键向量
        v = v_proj(x)  # 生成值向量

        # 计算注意力分数
        attention_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(hidden_size)  # 计算缩放点积注意力分数
        attention_probs = F.softmax(attention_scores, dim=-1)  # 对注意力分数应用softmax归一化

        # 应用注意力
        context_layer = torch.matmul(attention_probs, v)  # 计算注意力加权和

        # 输出投影
        attention_output = out_proj(context_layer)  # 通过输出投影层

        # 最后通过线性层
        return self.fc(attention_output + x)  # 添加残差连接并返回结果

# 定义 CLIP 模型
class CLIP(nn.Module):  # 定义CLIP模型类，继承自nn.Module
    """
    带完整 ViT 图像编码器的 CLIP 对比学习模型。

    将基于 ViT 的图像编码器和带自注意力的文本编码器组合成完整的 CLIP 框架：
    1. ViT 图像编码器：PatchEmbedding + 多层 Transformer 块，输出 [CLS] 全局特征
    2. 自注意力文本编码器：对文本特征建模后线性映射到嵌入空间
    3. 双向 L2 归一化 + 温度缩放 → 相似度矩阵
    该版本贴近真实 CLIP 模型结构，用于深入理解多模态对比学习的完整流程。
    """

    def __init__(self, embed_dim):  # 初始化方法，接收嵌入维度参数
        """
        初始化 CLIP 模型。

        参数：
            embed_dim (int): 共享嵌入空间的维度大小，同时也是 ViT 图像编码器的输出维度，
                             图像和文本特征在此维度下对齐，用于余弦相似度计算。
        """
        super(CLIP, self).__init__()  # 调用父类初始化方法
        self.image_encoder = ImageEncoder(embed_dim)  # 创建图像编码器实例
        self.text_encoder = TextEncoder(embed_dim)  # 创建文本编码器实例
        self.logit_scale = nn.Parameter(torch.ones([]) * 2.6592)  # 可学习的温度参数，初始化为2.6592

    def forward(self, images, texts):  # 前向传播方法，接收图像和文本输入
        """
        前向传播：计算图像与文本的跨模态相似度矩阵。

        参数：
            images (torch.Tensor): 输入图像张量，形状 (batch_size, 3, 224, 224)。
                                   - batch_size: 批次中的样本数量（即论文中的N）
                                   - 3: RGB 三通道
                                   - 224×224: 图像分辨率
            texts (torch.Tensor): 输入文本特征，形状 (batch_size, 512)。
                                  - batch_size: 批次中的样本数量（与图像数量相同）
                                  - 512: 文本编码器输入维度

        返回：
            tuple(torch.Tensor, torch.Tensor):
                - logits_per_image: 图像对所有文本的相似度得分，形状 (batch_size, batch_size)，
                                    第i行第j列表示第i张图像与第j段文本的相似度，对角线为正样本对。
                - logits_per_text: 文本对所有图像的相似度得分，形状 (batch_size, batch_size)，
                                   为 logits_per_image 的转置矩阵。
        """
        # 编码图像和文本
        image_features = self.image_encoder(images)  # 调用ViT图像编码器，返回 Tensor，形状 (batch_size, embed_dim)：取[CLS]token作为图像全局特征
        text_features = self.text_encoder(texts)  # 调用自注意力文本编码器，返回 Tensor，形状 (batch_size, embed_dim)：文本的上下文感知嵌入向量

        # 归一化特征：使每个特征向量的L2模长=1，余弦相似度等价于点积
        image_features = F.normalize(image_features, dim=-1)  # 沿嵌入维度做L2归一化，形状不变仍为 (batch_size, embed_dim)，每行向量的L2范数=1
        text_features = F.normalize(text_features, dim=-1)  # 沿嵌入维度做L2归一化，形状不变仍为 (batch_size, embed_dim)，每行向量的L2范数=1

        # 计算相似度矩阵：logit_scale放大余弦相似度，使分布更尖锐，便于分类
        logit_scale = self.logit_scale.exp()  # 对可学习温度参数取指数，返回标量 Tensor；初始值 exp(2.6592)≈14.3，训练后约为100
        logits_per_image = logit_scale * image_features @ text_features.t()  # 矩阵乘法计算图文余弦相似度并缩放，返回 Tensor，形状 (batch_size, batch_size)：第[i,j]元素=第i张图像与第j段文本的相似度得分
        logits_per_text = logits_per_image.t()  # 转置得到文本→图像方向的相似度矩阵，返回 Tensor，形状 (batch_size, batch_size)：第[j,i]元素=第j段文本与第i张图像的相似度得分

        return logits_per_image, logits_per_text  # 返回 tuple(Tensor,Tensor)：图像→文本得分矩阵(batch,batch) 和 文本→图像得分矩阵(batch,batch)

# 定义对比损失函数
def clip_loss(logits_per_image, logits_per_text):  # 定义CLIP损失函数，接收两个相似度矩阵
    """
    计算CLIP对比损失（InfoNCE Loss / 对称交叉熵损失）。

    对图像→文本和文本→图像两个方向分别计算交叉熵损失，然后取平均作为总损失。
    标签为对角线索引 (0, 1, 2, ..., N-1)，因为第i张图像与第i段文本互为正样本对。

    参数：
        logits_per_image (torch.Tensor): 图像对所有文本的相似度得分，形状 (batch_size, batch_size)。
                                         - 第i行第j列：第i张图像与第j段文本的相似度分数
                                         - 对角线元素为正样本对，其余为负样本对
        logits_per_text (torch.Tensor): 文本对所有图像的相似度得分，形状 (batch_size, batch_size)，
                                        即 logits_per_image 的转置矩阵。

    返回：
        torch.Tensor: 标量张量，CLIP对比损失值（图像→文本损失与文本→图像损失的平均值）。
    """
    # 计算图像到文本的损失
    labels = torch.arange(logits_per_image.size(0)).to(logits_per_image.device)  # 生成标签 [0,1,...,N-1]，类型 Tensor(long)，形状 (batch_size,)；第i个值=i表示第i张图像的正样本是第i段文本
    loss_image = F.cross_entropy(logits_per_image, labels)  # 以每行（图像）的所有文本得分为logit、labels为目标类别计算交叉熵，返回标量 Tensor（图像→文本方向的平均损失）

    # 计算文本到图像的损失
    loss_text = F.cross_entropy(logits_per_text, labels)  # 以每行（文本）的所有图像得分为logit、labels为目标类别计算交叉熵，返回标量 Tensor（文本→图像方向的平均损失）

    # 总损失：对两个方向取均值，实现对称对比损失
    total_loss = (loss_image + loss_text) / 2  # 两方向损失求平均，返回标量 Tensor，这正是论文公式中的对称 InfoNCE 损失
    return total_loss  # 返回标量 Tensor，即最终的CLIP对比损失值，用于反向传播更新参数

# 示例数据
batch_size = 4  # 批次大小（int）：一次前向传播包含4组图文对，即论文中的 N=4
embed_dim = 768  # 共享嵌入空间维度（int）：对应ViT-B/16的输出维度，图像和文本特征均被映射到768维空间
images = torch.randn(batch_size, 3, 224, 224)  # 随机生成图像张量，类型 Tensor，形状 (4, 3, 224, 224)：4张224×224 RGB图像（替代真实图像数据）
texts = torch.randn(batch_size, 512)    # 随机生成文本特征张量，类型 Tensor，形状 (4, 512)：4段文本，每段512维特征（替代真实Tokenizer+Transformer输出）

# 初始化模型
model = CLIP(embed_dim)  # 创建CLIP模型实例，返回 CLIP 对象，嵌入维度为768，内含ViT图像编码器和自注意力文本编码器

# 前向传播
logits_per_image, logits_per_text = model(images, texts)  # 执行模型前向传播，返回两个 Tensor：logits_per_image 形状(4,4)、logits_per_text 形状(4,4)，均为图文相似度得分矩阵
print("logits_per_image:", logits_per_image.shape)  # 打印图像→文本相似度矩阵的形状，期望输出 torch.Size([4, 4])：4张图像×4段文本
print("logits_per_text:", logits_per_text.shape)  # 打印文本→图像相似度矩阵的形状，期望输出 torch.Size([4, 4])：4段文本×4张图像
# 计算损失
loss = clip_loss(logits_per_image, logits_per_text)  # 计算对称InfoNCE损失，返回标量 Tensor，值约为 log(4)≈1.386（随机初始化时各候选均等概率）
print("Loss:", loss.item())  # 打印标量损失值（Python float），.item()将标量Tensor转为Python原生浮点数

logits_per_image: torch.Size([4, 4])
logits_per_text: torch.Size([4, 4])
Loss: 1.3869919776916504


## 四、CLIP 架构特性：双塔结构与无交叉注意力

### 4.1 CLIP 有交叉注意力（Cross-Attention）吗？

**结论：CLIP 没有交叉注意力。**

从前面打印的模型结构可以直接验证这一点：

```
CLIPModel(
  (text_model): CLIPTextTransformer
    (encoder): CLIPEncoder
      (0-11): 12 x CLIPEncoderLayer
        (self_attn): CLIPSdpaAttention   ← 只有自注意力，无交叉注意力

  (vision_model): CLIPVisionTransformer
    (encoder): CLIPEncoder
      (0-11): 12 x CLIPEncoderLayer
        (self_attn): CLIPSdpaAttention   ← 只有自注意力，无交叉注意力
)
```

每一层只有 `self_attn`（自注意力），文本编码器和图像编码器**完全独立**，在各自内部只做模态内部的自注意力，两个编码器之间没有任何交叉注意力模块。

---

### 4.2 CLIP 的图文交互方式：双塔结构

CLIP 的图文"交互"不发生在编码器内部，而是发生在**编码器之外**，通过以下方式实现：

```
图像 (224×224×3)
    → ViT 图像编码器（12 层自注意力，内部只看图像）
    → visual_projection（768 → 512）
    → L2 归一化
    → image_embeds  (N, 512)
                            ↘
                         矩阵点积 → 相似度矩阵 (N, M)
                            ↗
    → L2 归一化
    → text_projection（512 → 512）
    → Text Transformer（12 层自注意力，内部只看文本）
文本 (token序列)
```

**核心流程：**

1. 图像和文本各自独立地通过自己的 Transformer 编码器（模态内自注意力）
2. 各自经过投影层映射到同一维度（512 维共享嵌入空间）
3. 做 L2 归一化，使每个向量长度为 1（余弦相似度 = 点积）
4. 通过矩阵乘法计算图文余弦相似度矩阵
5. 乘以温度参数 `logit_scale`，得到最终 logits

这种架构称为**双塔结构（Dual Encoder / Two-Tower）**，两塔独立编码，只在最后的嵌入空间中"碰面"。

### 4.3 双塔结构的优缺点

| 维度 | 说明 |
|---|---|
| **优点：推理速度快** | 图像和文本的特征可以分别**提前计算并缓存**，检索时只需做向量点积，适合大规模图文检索（如以图搜图） |
| **优点：可扩展性强** | 4 亿图文对的大规模对比预训练成为可能，训练效率高 |
| **缺点：细粒度理解弱** | 由于两个模态在编码过程中完全独立，模型无法在内部进行细粒度的图文对齐（如"图中左边的红色物体"这类局部关联） |
| **缺点：交互发生太晚** | 图文交互只发生在最后的单一向量之间，丢失了序列级别的位置和局部信息 |

---

### 4.4 与 ALBEF 的架构对比

| 特性 | CLIP | ALBEF |
|---|---|---|
| **整体架构** | 双塔（Dual Encoder） | 单流融合（Multimodal Encoder） |
| **图文交互位置** | 编码器外（相似度矩阵，最后一步） | 编码器内（多模态编码器中的交叉注意力层） |
| **交叉注意力** | ❌ 无 | ✅ 有（文本 Query 注意到图像 Key/Value） |
| **图文对齐粒度** | 粗粒度（整体向量对齐） | 细粒度（token 级别对齐） |
| **推理速度** | 快（特征可预计算） | 慢（每次推理需联合编码） |
| **细粒度理解能力** | 较弱 | 较强（如 VQA、图文推理） |
| **适合任务** | 大规模零样本图文检索/分类 | VQA、图像描述、图文推理等需要细粒度理解的任务 |

**关键区别示意：**

```
CLIP（双塔，无交叉注意力）：
  [Image Encoder] ──→ image_vec ──┐
                                    ├── dot product → score
  [Text  Encoder] ──→  text_vec ──┘

ALBEF（融合，有交叉注意力）：
  [Image Encoder] ──→ image_tokens ──→ 作为 Key/Value ──┐
                                                           ├── Cross-Attention → 融合特征
  [Text  Encoder] ──→  text_tokens ──→  作为 Query    ──┘
```

> **总结**：CLIP 通过双塔结构和对比学习实现了高效的大规模图文预训练，但代价是放弃了细粒度的跨模态交互。ALBEF 等后续工作正是为了弥补这一缺陷，在两个独立编码器之后引入带交叉注意力的多模态融合编码器，兼顾检索效率与理解能力。